In [ ]:
import os, json, glob
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit

dbutils.widgets.text("catalog_name", "main")
dbutils.widgets.text("raw_schema", "iot_raw")
dbutils.widgets.text("landing_base", "")
dbutils.widgets.text("ingest_date", "")
dbutils.widgets.text("autoloader_state_base", "")
dbutils.widgets.text("config_dir", "")

catalog = dbutils.widgets.get("catalog_name")
raw_schema = dbutils.widgets.get("raw_schema")
landing_base = dbutils.widgets.get("landing_base").rstrip("/")          # -> .../iot
ingest_date = dbutils.widgets.get("ingest_date")                       # -> 20260227
state_base = dbutils.widgets.get("autoloader_state_base").rstrip("/")  # -> .../unitycatalog/pipeline_state/...
config_dir = dbutils.widgets.get("config_dir")

if not ingest_date:
    ingest_date = spark.sql("select date_format(current_timestamp(), 'yyyyMMdd') as d").collect()[0]["d"]

cfg_paths = sorted(glob.glob(os.path.join(config_dir, "*.json")))
datasets = []
for p in cfg_paths:
    with open(p, "r") as f:
        datasets.append(json.load(f))

def to_schema_hints(cfg: dict) -> str:
    parts = []
    for c in cfg["columns"]:
        spark_type = c.get("source_type") or c["type"]
        parts.append(f"{c['name']} {spark_type.upper()}")
    return ", ".join(parts)

for cfg in datasets:
    name = cfg["name"]

    # Assumption: the daily folder contains all new files, and file names start with dataset name
    # Example: .../iot/20260227/power_components.csv
    file_glob = f"{name}*.csv"
    src_path = f"{landing_base}/{ingest_date}/{file_glob}"

    raw_table = f"{catalog}.{raw_schema}.{name}_raw"

    schema_loc = f"{state_base}/_schemas/{name}"
    checkpoint_loc = f"{state_base}/_checkpoints/{name}"

    schema_hints = to_schema_hints(cfg)
    delimiter = cfg.get("csv_options", {}).get("delimiter", "\t")
    header = str(cfg.get("csv_options", {}).get("header", True)).lower()

    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_loc)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .option("cloudFiles.schemaHints", schema_hints)
        .option("header", header)
        .option("sep", delimiter)
        .load(src_path)
        .withColumn("_ingest_ts", current_timestamp())
        .withColumn("_ingest_date", lit(ingest_date))
        .withColumn("_source_file", F.col("_metadata.file_path"))
        # Optional (nice for audit/debug):
        .withColumn("_source_file_mod_time", F.col("_metadata.file_modification_time"))
        .withColumn("_source_file_size", F.col("_metadata.file_size"))
    )

    (
        df.writeStream
        .option("checkpointLocation", checkpoint_loc)
        .trigger(availableNow=True)
        .toTable(raw_table)
    )

print(f"Raw ingestion complete into {catalog}.{raw_schema}.")